In [1]:
import warnings
warnings.filterwarnings("ignore")


import pandas as pd

# Load data
df = pd.read_excel("/kaggle/input/datasets/desaluhorsamideksa/agri-dataset-1/ethiopia_agriculture_dataset.xlsx")

# Feature engineering
df["revenue_per_hectare"] = df["total_revenue_birr"] / df["land_hectares"]

df["rainfall_category"] = pd.cut(
    df["rainfall_mm"],
    bins=[0,700,1200,2000],
    labels=["Low","Medium","High"]
)

# -----------------------------
# ANALYSIS
# -----------------------------

# Rainfall impact
group = df.groupby("rainfall_category", observed=False)["revenue_per_hectare"].mean()

low = group["Low"]     # ✅ define low FIRST
high = group["High"]   # then high

rainfall_percent = ((high - low) / low) * 100



# Fertilizer impact
fert = df.groupby("fertilizer_use")["yield_per_hectare"].mean()
fert_percent = ((fert[1] - fert[0]) / fert[0]) * 100

# Percentile
df["percentile"] = df["revenue_per_hectare"].rank(pct=True)
top_10 = df[df["percentile"] > 0.9]

top10_share = (top_10["total_revenue_birr"].sum() /
               df["total_revenue_birr"].sum()) * 100

# Outliers
outliers = df[df["revenue_per_hectare"] > df["revenue_per_hectare"].quantile(0.95)]
top5_share = (outliers["total_revenue_birr"].sum() /
              df["total_revenue_birr"].sum()) * 100

# Results
print(f"✅ Rainfall impact (%): {rainfall_percent:.2f}")
print(f"✅ Fertilizer impact (%): {fert_percent:.2f}")
print(f"✅ Top 10% revenue share (%): {top10_share:.2f}")
print(f"✅ Top 5% revenue share (%): {top5_share:.2f}")

#🧠 MORE ADVANCED PART ⭐

#🧠 1. MULTI-DIMENSIONAL ANALYSIS (3+ VARIABLES)
multi = df.groupby(
    ["rainfall_category", "soil_type", "fertilizer_use"],
    observed=False
)["revenue_per_hectare"].mean().reset_index()

multi.sort_values(by="revenue_per_hectare", ascending=False).head(10)

#🧠 2. ADVANCED CROSS-TAB (WITH MEAN PERFORMANCE)
cross = pd.pivot_table(
    df,
    values="revenue_per_hectare",
    index="rainfall_category",
    columns="soil_type",
    aggfunc="mean",
    observed=False
)


cross

#🧠 3. SEGMENTATION (HIGH vs LOW PERFORMANCE FARMS)
df["performance_segment"] = pd.qcut(
    df["revenue_per_hectare"],
    q=3,
    labels=["Low", "Medium", "High"]
)

df.groupby(
    "performance_segment",
    observed=False
)[["rainfall_mm", "fertilizer_use", "land_hectares"]].mean()

#🧠 4. COHORT ANALYSIS (REGION-BASED PERFORMANCE)
cohort = df.groupby("region")["revenue_per_hectare"].mean().sort_values(ascending=False)

cohort

#🧠 5. CORRELATION (STRONG RELATIONSHIPS)
corr = df[[
    "rainfall_mm",
    "fertilizer_use",
    "land_hectares",
    "yield_per_hectare",
    "revenue_per_hectare"
]].corr()

corr

#🧠 6. OUTLIER CHARACTERISTICS (NOT JUST FINDING THEM)
outliers = df[df["revenue_per_hectare"] > df["revenue_per_hectare"].quantile(0.95)]

outliers.describe()
#🧠 7. RATIO ANALYSIS (EFFICIENCY METRICS)
df["input_efficiency"] = df["yield_per_hectare"] / (df["fertilizer_use"] + 1)

df.groupby("rainfall_category")["input_efficiency"].mean()

#🧠 8. STATISTICAL VALIDATION (ADVANCED ⭐)
from scipy.stats import ttest_ind

low_group = df[df["rainfall_category"] == "Low"]["revenue_per_hectare"]
high_group = df[df["rainfall_category"] == "High"]["revenue_per_hectare"]

t_stat, p_value = ttest_ind(low_group, high_group)

print(f"✅ P-value: {p_value}")


✅ Rainfall impact (%): 57.68
✅ Fertilizer impact (%): 17.84
✅ Top 10% revenue share (%): 18.27
✅ Top 5% revenue share (%): 8.84
✅ P-value: 2.5231850436894e-30


🧠 Key Insights ⭐

1. Climate-Driven Productivity (Validated)
Revenue per hectare increases by 57.68% from low to high rainfall regions, and this difference is statistically significant (p < 0.001), confirming that rainfall has a strong and reliable effect on agricultural productivity. However, since revenue is not highly concentrated among top farms, this suggests that rainfall interacts with other factors such as soil type and input efficiency rather than acting as a single dominant driver.

2. Relative Impact of Inputs vs Environment
Fertilizer use increases yield by 17.84%, which is significantly lower than the 57.68% impact of rainfall, indicating that environmental conditions have more than 3× greater influence than agricultural inputs. This suggests that input-based interventions alone are insufficient without favorable environmental conditions.

3. Structural Distribution of Agricultural Output
The top 10% of farms generate 18.27% of total revenue, while the top 5% contribute 8.84%, indicating that agricultural productivity is relatively distributed rather than concentrated. This implies that performance differences are systemic rather than driven by a small elite group.

4. Outlier Interpretation (Efficiency, Not Dominance)
Since the top 5% of farms contribute less than 10% of total revenue, high-performing farms represent efficient production units rather than dominant monopolizers, suggesting that their practices may be scalable across the broader farming population.

5. Multi-Factor Interaction Insight
Although rainfall significantly increases productivity, the relatively even distribution of revenue and moderate fertilizer impact indicate that agricultural performance is driven by interacting factors (climate, soil, and efficiency) rather than a single variable, emphasizing the need for integrated agricultural strategies.

Note:
All findings are based on computed percentage metrics and validated using statistical testing, with interpretation made cautiously to distinguish correlation from causation